### OpenAI API key verification (from `.env`)

This notebook checks that `OPENAI_API_KEY` is loaded from a project-local `.env` file (via `python-dotenv`) and prints a masked preview.

### Prereqs

1. Create a file named `.env` in the project root (same folder as this notebook).
2. Add a line like:

```dotenv
OPENAI_API_KEY=...
```

This notebook will **not** print your full key; it only prints a masked preview.

In [ ]:
import hashlib
import json
import os
import time
from datetime import datetime, timezone
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI
from PyPDF2 import PdfReader

_ = (hashlib, json, os, time, datetime, timezone, Any, load_dotenv, OpenAI, PdfReader)

In [ ]:
# Load variables from a local .env into process environment
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Create a .env file in the project root with OPENAI_API_KEY=..."
    )

# Print a masked preview (never print the full key)
masked = api_key[:7] + "…" + api_key[-4:] if len(api_key) > 12 else "(set, short key)"
print("OPENAI_API_KEY loaded:", masked)

In [ ]:
# --- Safe Vector Store Ingestion ---
import os
import json
import time

VECTOR_STORE_NAME = "supreme court cases"
DOCS_DIR = "docs"
EMBEDDINGS_DIR = "embeddings"
VECTOR_STORE_DIR = "vector_store"
STORE_SUMMARY_PATH = os.path.join(EMBEDDINGS_DIR, "vector_store.supreme-court-cases.json")

os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
os.makedirs(VECTOR_STORE_DIR, exist_ok=True)

# 1. Initialize Client
client = OpenAI(api_key=api_key)

# 2. Get or Create Vector Store
existing_id = None
for vs in client.vector_stores.list(limit=100):
    if vs.name == VECTOR_STORE_NAME:
        existing_id = vs.id
        break

if existing_id:
    vector_store_id = existing_id
    print(f"Using existing vector store: {vector_store_id}")
else:
    vs = client.vector_stores.create(name=VECTOR_STORE_NAME)
    vector_store_id = vs.id
    print(f"Created new vector store: {vector_store_id}")

# Save reference for the Streamlit app
with open(STORE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump({"vector_store_id": vector_store_id}, f, indent=2)

# 3. Analyze Existing Files (Safe Logic to Prevent Duplicates)
existing_filenames = {}
for vs_file in client.vector_stores.files.list(vector_store_id=vector_store_id, limit=100):
    f_id = getattr(vs_file, "file_id", getattr(vs_file, "id", None))
    if f_id:
        try:
            fobj = client.files.retrieve(f_id)
            fn = getattr(fobj, "filename", "").strip()
            if fn:
                existing_filenames[fn.lower()] = f_id
        except Exception:
            continue

print(f"Vector Store currently contains {len(existing_filenames)} files.")

# 4. Upload Missing Files from docs/
if not os.path.exists(DOCS_DIR):
    os.makedirs(DOCS_DIR)

found_pdfs = [f for f in os.listdir(DOCS_DIR) if f.lower().endswith(".pdf")]
if not found_pdfs:
    print(f"No PDFs found in {DOCS_DIR}/ folder. Please add your documents!")
else:
    for name in found_pdfs:
        pdf_path = os.path.join(DOCS_DIR, name)
        stem, _ = os.path.splitext(name)
        guard_path = os.path.join(VECTOR_STORE_DIR, f"{stem}.vector_store_record.json")
        
        # Check against remote duplicates
        if name.lower() in existing_filenames:
            existing_file_id = existing_filenames[name.lower()]
            print(f"SKIP (Already Uploaded): {name}")
            
            # Recreate local guard file if it was cleared
            record = {
                "file_id": existing_file_id, 
                "vector_store_id": vector_store_id, 
                "document_filename": name,
                "note": "Skipped duplicate upload"
            }
            with open(guard_path, "w", encoding="utf-8") as f:
                json.dump(record, f, indent=2)
            continue
            
        # Perform Upload
        print(f"UPLOADING: {name} ...")
        with open(pdf_path, "rb") as f:
            uploaded = client.files.create(file=f, purpose="assistants")
        
        # Attach to Vector Store
        client.vector_stores.files.create(vector_store_id=vector_store_id, file_id=uploaded.id)
        print(f"SUCCESS: Uploaded {name} -> File ID: {uploaded.id}")
        
        # Create Local Record
        record = {
            "file_id": uploaded.id, 
            "vector_store_id": vector_store_id, 
            "document_filename": name,
            "note": "Fresh upload"
        }
        with open(guard_path, "w", encoding="utf-8") as f:
            json.dump(record, f, indent=2)

    print("Ingestion complete!")